In [ ]:
import regex as re
from collections import Counter
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

In [ ]:
type Tokens = list[str]


def clean_text(text: str) -> str:
    text = text.lower()
    text = text.replace('"', "")  # remove quotes
    text = re.sub(r"\(.*?\)", "", text)  # remove everything inside of brackets
    text = re.sub(r"(?<=\w)'\w+", "", text)  # remove everything after an apostrophe
    text = re.sub(r"\W", " ", text)  # split at other special characters
    text = re.sub(r"\s{2,}", " ", text)  # collaps multiple spaces
    return text.strip()


def get_sentences(text: str) -> list[str]:
    text = re.sub(r"(?:[!?]|\.(?!\d))+", ".", text)
    return text.split(".")


def tokenize(text: str) -> Tokens:
    return text.split(" ")


def extract_groups(sentence_tokens: Tokens, context_size: int) -> list[Tokens]:
    if len(sentence_tokens) < context_size * 2 + 1:
        return []

    return np.lib.stride_tricks.sliding_window_view(
        sentence_tokens, context_size * 2 + 1
    )

#### Tokenize & Count Unique Tokens

In [ ]:
parquet = pq.ParquetFile("data/000_00000.parquet")

token_counts = Counter()
max_tokens = 0
for batch in parquet.iter_batches(
    batch_size=10_000,
):
    for text in batch.column("text").to_pylist():
        text = clean_text(text)
        sentences = get_sentences(text)
        tokens = 0
        for sentence_tokens in sentences:
            sentence_tokens = tokenize(sentence_tokens)
            token_counts.update(sentence_tokens)
            tokens += len(sentence_tokens)

        if tokens > max_tokens:
            max_tokens = tokens

#### Remove tokens below MIN_OCCURANCES

In [ ]:
token_to_id: dict[str, int] = dict()
id_to_token: dict[int, str] = dict()

In [ ]:
MIN_OCCURANCES = 20

i = 0
for token, count in token_counts.items():
    if count < MIN_OCCURANCES:
        continue

    token_to_id[token] = i
    id_to_token[i] = token
    i += 1
    

In [ ]:
import pickle

pickle.dump(token_to_id, Path("token_to_id.pkl").open("wb"))
pickle.dump(id_to_token, Path("id_to_token.pkl").open("wb"))

#### Build Groups

In [ ]:
from time import time

OUT_FILE = Path("groups.csv").open("a", encoding="utf-8")
BATCH_SIZE = 10_000


def is_valid_group(group: Tokens) -> bool:
    for token in group:
        if token_counts[token] < MIN_OCCURANCES:
            return False
    return True


n_batches = parquet.metadata.num_rows // BATCH_SIZE
for i, batch in enumerate(
    parquet.iter_batches(
        batch_size=BATCH_SIZE,
    )
):
    print(f"{i + 1} / {n_batches} ({n_batches - i} remaining)", end=" ")
    start = time()
    for text in batch.column("text").to_pylist():
        text = clean_text(text)
        sentences = get_sentences(text)
        for sentence_tokens in sentences:
            sentence_tokens = tokenize(sentence_tokens)
            groups = extract_groups(sentence_tokens, 4)
            groups = filter(is_valid_group, groups)
            OUT_FILE.write(
                "\n".join(
                    [
                        ",".join(map(str, map(lambda x: token_to_id[x], group)))
                        for group in groups
                    ]
                )
                + "\n"
            )
    end = time()
    print(f"{end - start:.2f}s")

In [ ]:
OUT_FILE.close()